In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, confusion_matrix, precision_score, recall_score
import joblib
import warnings
warnings.filterwarnings('ignore')

print("All imports loaded")

All imports loaded


In [ ]:
fraud_df = pd.read_csv('../data/processed/fraud_data_processed.csv')
print(f"Shape: {fraud_df.shape}")
print(f"Fraud rate: {fraud_df['class'].mean()*100:.4f}%")

Shape: (151112, 20)
Fraud rate: 9.3646%


In [ ]:
categorical_cols = ['source', 'browser', 'sex', 'country']
fraud_df = pd.get_dummies(fraud_df, columns=categorical_cols, drop_first=True)

feature_cols = ['purchase_value', 'age', 'hour_of_day', 'day_of_week',
                'time_since_signup_hours', 'new_user_1h', 'users_per_device']
feature_cols += [col for col in fraud_df.columns if col.startswith(('source_', 'browser_', 'sex_', 'country_'))]

X = fraud_df[feature_cols].fillna(0)
y = fraud_df['class']

print(f"Features: {X.shape[1]}")
print(f"Samples: {X.shape[0]}")

Features: 14
Samples: 151112


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape[0]} samples")
print(f"Test: {X_test.shape[0]} samples")
print(f"Train fraud rate: {y_train.mean()*100:.4f}%")

Train: 120889 samples
Test: 30223 samples
Train fraud rate: 9.3648%


In [ ]:
print("BEFORE SMOTE:")
print(f"  Class 0: {sum(y_train==0)}")
print(f"  Class 1: {sum(y_train==1)}")

smote = SMOTE(random_state=42, sampling_strategy=0.5)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print("\nAFTER SMOTE:")
print(f"  Class 0: {sum(y_train_resampled==0)}")
print(f"  Class 1: {sum(y_train_resampled==1)}")

BEFORE SMOTE:
  Class 0: 109568
  Class 1: 11321

AFTER SMOTE:
  Class 0: 109568
  Class 1: 54784


In [ ]:
models = {
    'Logistic Regression': LogisticRegression(random_state=42, class_weight='balanced', max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, class_weight='balanced'),
    'XGBoost': XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, scale_pos_weight=5, eval_metric='logloss')
}

results = {}
print("Training models...\n")

for name, model in models.items():
    model.fit(X_train_resampled, y_train_resampled)
    y_pred = model.predict(X_test)
    f1 = f1_score(y_test, y_pred)
    results[name] = {'model': model, 'f1': f1}
    print(f"{name}: F1-Score = {f1:.4f}")

Training models...

Logistic Regression: F1-Score = 0.6117
Random Forest: F1-Score = 0.6193
XGBoost: F1-Score = 0.6154


In [ ]:
print("\n" + "="*50)
print("MODEL COMPARISON")
print("="*50)

for name, result in sorted(results.items(), key=lambda x: x[1]['f1'], reverse=True):
    print(f"{name:<20} F1: {result['f1']:.4f}")

best_name = max(results, key=lambda x: results[x]['f1'])
best_f1 = results[best_name]['f1']
print(f"\n BEST MODEL: {best_name} (F1 = {best_f1:.4f})")


MODEL COMPARISON
Random Forest        F1: 0.6193
XGBoost              F1: 0.6154
Logistic Regression  F1: 0.6117

 BEST MODEL: Random Forest (F1 = 0.6193)


In [ ]:
joblib.dump(results[best_name]['model'], '../models/best_fraud_model.pkl')
joblib.dump(scaler, '../models/scaler.pkl')
print(" Model saved to ../models/best_fraud_model.pkl")
print(" Scaler saved to ../models/scaler.pkl")

 Model saved to ../models/best_fraud_model.pkl
 Scaler saved to ../models/scaler.pkl


In [ ]:
from google.colab import files
print("Please upload Fraud_Data.csv from your computer")
uploaded = files.upload()

Please upload Fraud_Data.csv from your computer


Saving Fraud_Data.csv to Fraud_Data.csv


In [ ]:
# COMPREHENSIVE EVALUATION METRICS - Upload Version
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             confusion_matrix, recall_score, precision_score,
                             f1_score, classification_report)
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import pandas as pd
import numpy as np
import io

print("="*60)
print("COMPREHENSIVE EVALUATION METRICS")
print("="*60)

# Get uploaded filename
file_name = list(uploaded.keys())[0]
fraud_df = pd.read_csv(io.BytesIO(uploaded[file_name]))
print(f"Data loaded: {fraud_df.shape}")

# Feature engineering
fraud_df['signup_time'] = pd.to_datetime(fraud_df['signup_time'])
fraud_df['purchase_time'] = pd.to_datetime(fraud_df['purchase_time'])
fraud_df['hour_of_day'] = fraud_df['purchase_time'].dt.hour
fraud_df['day_of_week'] = fraud_df['purchase_time'].dt.dayofweek
fraud_df['time_since_signup_hours'] = (fraud_df['purchase_time'] - fraud_df['signup_time']).dt.total_seconds() / 3600
fraud_df['new_user_1h'] = (fraud_df['time_since_signup_hours'] <= 1).astype(int)

device_users = fraud_df.groupby('device_id')['user_id'].nunique().to_dict()
fraud_df['users_per_device'] = fraud_df['device_id'].map(device_users).fillna(1)

fraud_df = pd.get_dummies(fraud_df, columns=['source', 'browser', 'sex'], drop_first=True)

feature_cols = ['purchase_value', 'age', 'hour_of_day', 'day_of_week',
                'time_since_signup_hours', 'new_user_1h', 'users_per_device']
feature_cols += [col for col in fraud_df.columns if col.startswith(('source_', 'browser_', 'sex_'))]

X = fraud_df[feature_cols].fillna(0)
y = fraud_df['class']

# Split and SMOTE
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
smote = SMOTE(random_state=42, sampling_strategy=0.5)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

# Train model
best_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, class_weight='balanced')
best_model.fit(X_train_resampled, y_train_resampled)

# Predict
y_pred = best_model.predict(X_test)
y_pred_proba = best_model.predict_proba(X_test)[:, 1]

# Calculate metrics
roc_auc = roc_auc_score(y_test, y_pred_proba)
pr_auc = average_precision_score(y_test, y_pred_proba)
f1 = f1_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
false_positive_rate = fp / (fp + tn) if (fp + tn) > 0 else 0
false_negative_rate = fn / (fn + tp) if (fn + tp) > 0 else 0

print("\n" + "="*60)
print("RANDOM FOREST RESULTS")
print("="*60)
print(f"ROC-AUC: {roc_auc:.4f}")
print(f"PR-AUC: {pr_auc:.4f}")
print(f"F1-Score: {f1:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"False Positive Rate: {false_positive_rate:.4f}")
print(f"False Negative Rate: {false_negative_rate:.4f}")

print("\n" + "="*60)
print("CONFUSION MATRIX")
print("="*60)
print(f"True Negatives: {tn} | False Positives: {fp}")
print(f"False Negatives: {fn} | True Positives: {tp}")

print("\n" + "="*60)
print("CLASSIFICATION REPORT")
print("="*60)
print(classification_report(y_test, y_pred, target_names=['Legit', 'Fraud']))

COMPREHENSIVE EVALUATION METRICS
Data loaded: (151112, 11)

RANDOM FOREST RESULTS
ROC-AUC: 0.8370
PR-AUC: 0.6936
F1-Score: 0.6655
Precision: 0.7565
Recall: 0.5940
False Positive Rate: 0.0197
False Negative Rate: 0.4060

CONFUSION MATRIX
True Negatives: 26852 | False Positives: 541
False Negatives: 1149 | True Positives: 1681

CLASSIFICATION REPORT
              precision    recall  f1-score   support

       Legit       0.96      0.98      0.97     27393
       Fraud       0.76      0.59      0.67      2830

    accuracy                           0.94     30223
   macro avg       0.86      0.79      0.82     30223
weighted avg       0.94      0.94      0.94     30223



In [ ]:
# HYPERPARAMETER TUNING - GridSearchCV
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

print("="*60)
print("HYPERPARAMETER TUNING")
print("="*60)

# Parameter grid
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, 15, 20],
    'min_samples_split': [2, 5]
}

# Grid search
grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42, class_weight='balanced'),
    param_grid,
    cv=3,
    scoring='f1',
    n_jobs=-1
)

grid_search.fit(X_train_resampled, y_train_resampled)

print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best CV F1-Score: {grid_search.best_score_:.4f}")

# Evaluate tuned model
tuned_model = grid_search.best_estimator_
y_pred_tuned = tuned_model.predict(X_test)
print(f"Test F1-Score (Tuned): {f1_score(y_test, y_pred_tuned):.4f}")

HYPERPARAMETER TUNING
Best Parameters: {'max_depth': 20, 'min_samples_split': 2, 'n_estimators': 200}
Best CV F1-Score: 0.8388
Test F1-Score (Tuned): 0.6550


In [14]:
# COMPLETE IMPROVEMENTS PACKAGE

# ============================================================
# CELL 1: COMPREHENSIVE EVALUATION METRICS
# ============================================================
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             confusion_matrix, recall_score, precision_score,
                             f1_score, classification_report)
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import pandas as pd
import numpy as np
import io
from google.colab import files

print("="*60)
print("STEP 1: UPLOAD DATA")
print("="*60)

print("Please upload Fraud_Data.csv from your computer")
uploaded = files.upload()
file_name = list(uploaded.keys())[0]
fraud_df = pd.read_csv(io.BytesIO(uploaded[file_name]))
print(f"Data loaded: {fraud_df.shape}")

# Feature engineering
fraud_df['signup_time'] = pd.to_datetime(fraud_df['signup_time'])
fraud_df['purchase_time'] = pd.to_datetime(fraud_df['purchase_time'])
fraud_df['hour_of_day'] = fraud_df['purchase_time'].dt.hour
fraud_df['day_of_week'] = fraud_df['purchase_time'].dt.dayofweek
fraud_df['time_since_signup_hours'] = (fraud_df['purchase_time'] - fraud_df['signup_time']).dt.total_seconds() / 3600
fraud_df['new_user_1h'] = (fraud_df['time_since_signup_hours'] <= 1).astype(int)

device_users = fraud_df.groupby('device_id')['user_id'].nunique().to_dict()
fraud_df['users_per_device'] = fraud_df['device_id'].map(device_users).fillna(1)

fraud_df = pd.get_dummies(fraud_df, columns=['source', 'browser', 'sex'], drop_first=True)

feature_cols = ['purchase_value', 'age', 'hour_of_day', 'day_of_week',
                'time_since_signup_hours', 'new_user_1h', 'users_per_device']
feature_cols += [col for col in fraud_df.columns if col.startswith(('source_', 'browser_', 'sex_'))]

X = fraud_df[feature_cols].fillna(0)
y = fraud_df['class']

# Split and SMOTE
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
smote = SMOTE(random_state=42, sampling_strategy=0.5)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

# Train model
best_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, class_weight='balanced')
best_model.fit(X_train_resampled, y_train_resampled)

# Predict
y_pred = best_model.predict(X_test)
y_pred_proba = best_model.predict_proba(X_test)[:, 1]

# Calculate metrics
roc_auc = roc_auc_score(y_test, y_pred_proba)
pr_auc = average_precision_score(y_test, y_pred_proba)
f1 = f1_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
false_positive_rate = fp / (fp + tn) if (fp + tn) > 0 else 0
false_negative_rate = fn / (fn + tp) if (fn + tp) > 0 else 0

print("\n" + "="*60)
print("RANDOM FOREST RESULTS")
print("="*60)
print(f"ROC-AUC: {roc_auc:.4f}")
print(f"PR-AUC: {pr_auc:.4f}")
print(f"F1-Score: {f1:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"False Positive Rate: {false_positive_rate:.4f}")
print(f"False Negative Rate: {false_negative_rate:.4f}")

print("\n" + "="*60)
print("CONFUSION MATRIX")
print("="*60)
print(f"True Negatives: {tn} | False Positives: {fp}")
print(f"False Negatives: {fn} | True Positives: {tp}")

print("\n" + "="*60)
print("CLASSIFICATION REPORT")
print("="*60)
print(classification_report(y_test, y_pred, target_names=['Legit', 'Fraud']))


# ============================================================
# CELL 2: FASTER HYPERPARAMETER TUNING
# ============================================================
from sklearn.model_selection import RandomizedSearchCV

print("\n" + "="*60)
print("STEP 2: HYPERPARAMETER TUNING")
print("="*60)

param_dist = {
    'n_estimators': [100, 150, 200],
    'max_depth': [10, 15, 20],
    'min_samples_split': [2, 5]
}

random_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, class_weight='balanced'),
    param_dist,
    n_iter=5,
    cv=3,
    scoring='f1',
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train_resampled, y_train_resampled)

print(f"Best Parameters: {random_search.best_params_}")
print(f"Best CV F1-Score: {random_search.best_score_:.4f}")

tuned_model = random_search.best_estimator_
y_pred_tuned = tuned_model.predict(X_test)
print(f"Test F1-Score (Tuned): {f1_score(y_test, y_pred_tuned):.4f}")


# ============================================================
# CELL 3: COMPREHENSIVE MODEL COMPARISON TABLE
# ============================================================
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

print("\n" + "="*60)
print("STEP 3: MODEL COMPARISON")
print("="*60)

models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=100, max_depth=6, random_state=42, eval_metric='logloss'),
    'Tuned Random Forest': tuned_model
}

results_list = []

for name, model in models.items():
    model.fit(X_train_resampled, y_train_resampled)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    results_list.append({
        'Model': name,
        'F1': f1_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_proba),
        'PR-AUC': average_precision_score(y_test, y_proba)
    })

comparison_df = pd.DataFrame(results_list)
print(comparison_df.to_string(index=False))

# Save to CSV
import os
os.makedirs('../reports', exist_ok=True)
comparison_df.to_csv('../reports/model_comparison.csv', index=False)
print("\n Saved: model_comparison.csv")

STEP 1: UPLOAD DATA
Please upload Fraud_Data.csv from your computer


Saving Fraud_Data.csv to Fraud_Data (2).csv
Data loaded: (151112, 11)

RANDOM FOREST RESULTS
ROC-AUC: 0.8370
PR-AUC: 0.6936
F1-Score: 0.6655
Precision: 0.7565
Recall: 0.5940
False Positive Rate: 0.0197
False Negative Rate: 0.4060

CONFUSION MATRIX
True Negatives: 26852 | False Positives: 541
False Negatives: 1149 | True Positives: 1681

CLASSIFICATION REPORT
              precision    recall  f1-score   support

       Legit       0.96      0.98      0.97     27393
       Fraud       0.76      0.59      0.67      2830

    accuracy                           0.94     30223
   macro avg       0.86      0.79      0.82     30223
weighted avg       0.94      0.94      0.94     30223


STEP 2: HYPERPARAMETER TUNING
Best Parameters: {'n_estimators': 200, 'min_samples_split': 2, 'max_depth': 15}
Best CV F1-Score: 0.8177
Test F1-Score (Tuned): 0.6647

STEP 3: MODEL COMPARISON


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


              Model       F1  Precision   Recall  ROC-AUC   PR-AUC
Logistic Regression 0.668854   0.877727 0.540283 0.786400 0.632683
      Random Forest 0.692325   0.934535 0.549823 0.837597 0.695434
            XGBoost 0.672330   0.842469 0.559364 0.828825 0.677521
Tuned Random Forest 0.664672   0.763170 0.588693 0.830356 0.680619

 Saved: model_comparison.csv


In [15]:
# CELL 4: COMPLETE SHAP VISUAL ARTIFACTS
import shap
import matplotlib.pyplot as plt

print("="*60)
print("STEP 4: SHAP VISUAL ARTIFACTS")
print("="*60)

X_sample = X_test[:100]
explainer = shap.TreeExplainer(tuned_model)
shap_values = explainer.shap_values(X_sample)

if isinstance(shap_values, list):
    shap_values_class = shap_values[1]
    expected_value = explainer.expected_value[1]
else:
    shap_values_class = shap_values
    expected_value = explainer.expected_value

print("Generating SHAP visualizations...")

try:
    shap.force_plot(expected_value, shap_values_class[0], X_sample.iloc[0],
                    feature_names=feature_cols, matplotlib=True, show=False)
    plt.title('SHAP Force Plot - Sample Transaction')
    plt.tight_layout()
    plt.savefig('../reports/shap_force_plot.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("Saved: shap_force_plot.png")
except Exception as e:
    print(f"Force plot skipped: {e}")

try:
    shap.summary_plot(shap_values_class, X_sample, feature_names=feature_cols, show=False)
    plt.title('SHAP Summary Plot')
    plt.tight_layout()
    plt.savefig('../reports/shap_summary_complete.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("Saved: shap_summary_complete.png")
except Exception as e:
    print(f"Summary plot skipped: {e}")

print("\nSHAP artifacts generation complete")

STEP 4: SHAP VISUAL ARTIFACTS
Generating SHAP visualizations...
Force plot skipped: In v0.20, force plot now requires the base value as the first parameter! Try shap.plots.force(explainer.expected_value, shap_values) or for multi-output models try shap.plots.force(explainer.expected_value[0], shap_values[..., 0]).
Saved: shap_summary_complete.png

SHAP artifacts generation complete
